# 03 — Model Development

Train the canonical model ladder:

| Tier        | Model                           | Purpose                              |
|-------------|---------------------------------|--------------------------------------|
| Trivial     | `DummyClassifier(most_frequent)`| Sanity baseline                       |
| Classical   | Decision Tree (depth=8)         | Interaction-capturing baseline       |
| Classical   | Logistic Regression (L2)        | Interpretable, GridSearch-tuned      |
| Advanced    | Random Forest (300–800 trees)   | RandomizedSearch-tuned               |
| Advanced    | XGBoost                         | BayesSearch-tuned (likely winner)    |
| Advanced    | MLP (2 hidden, dropout)         | Neural-network box-check             |

Every tunable model is wrapped with `CalibratedClassifierCV(method='isotonic', cv='prefit')` because the EV math downstream requires *calibrated* probabilities, not just rankings. The scoring function passed to every search is the project's custom **`profit_scorer`** — i.e. we tune for per-flight ROI under τ\*(T, d), not for ROC-AUC or F1.

This notebook is reproducible from a fresh checkout: if `data/processed/flights.parquet` does not exist, the prep cell below builds it from real BTS files in `data/raw/` (preferred) or from the synthetic generator (fallback).

In [1]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
np.seterr(all="ignore")

ROOT = Path.cwd()
if (ROOT / "src").exists():
    sys.path.insert(0, str(ROOT))
elif (ROOT.parent / "src").exists():
    sys.path.insert(0, str(ROOT.parent))

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 60)


## Data preparation — self-healing

If a previously-built `data/processed/flights.parquet` is on disk we use it. Otherwise we build it now. Real BTS parquet/CSV files in `data/raw/` are preferred; if there are none, the synthetic generator fills in. The synthetic path is what the smoke test and CI use, so the pipeline is guaranteed to run from any checkout.

In [2]:
import os
from src.config import PROCESSED_DIR, RAW_DIR
from src.data.ec261 import label_eligible_delay
from src.data.loaders import (
    add_ticket_price, augment_with_aircraft, augment_with_weather,
    load_bts, load_faa_registry, prepare_modelling_frame,
)

FLIGHTS_PARQUET = PROCESSED_DIR / "flights.parquet"

real_files = list(RAW_DIR.glob("bts_*.parquet")) + list(RAW_DIR.glob("bts_*.csv"))
USING_REAL_DATA = bool(real_files)

if FLIGHTS_PARQUET.exists():
    print(f"Loading existing {FLIGHTS_PARQUET}")
    df = pd.read_parquet(FLIGHTS_PARQUET)
else:
    if USING_REAL_DATA:
        print(f"Building flights.parquet from {len(real_files)} real BTS file(s)...")
        df = load_bts(fallback="synthetic")
    else:
        n_synthetic = int(os.environ.get("N_SYNTHETIC", 80_000))
        print(f"No real BTS files found. Building flights.parquet from synthetic data (n={n_synthetic:,})...")
        df = load_bts(fallback="synthetic", n_synthetic=n_synthetic, force_synthetic=True)
    df = augment_with_aircraft(df, load_faa_registry())
    df = augment_with_weather(df)
    df = prepare_modelling_frame(df)
    df = add_ticket_price(df, seed=42)
    df["y_eligible_delay"] = label_eligible_delay(df).to_numpy()
    df.to_parquet(FLIGHTS_PARQUET, index=False)
    print(f"Wrote {FLIGHTS_PARQUET} (rows={len(df):,})")

print(f"\nDataset shape: {df.shape}")
print(f"Date range:    {df['FL_DATE'].min().date()} → {df['FL_DATE'].max().date()}")
print(f"Base rate (y_eligible_delay): {df['y_eligible_delay'].mean():.4%}")
print(f"Data source:   {'REAL BTS' if USING_REAL_DATA else 'SYNTHETIC (smoke-test grade)'}")


Loading existing /home/claude/Flight-delay-predictor/data/processed/flights.parquet

Dataset shape: (14701, 29)
Date range:    2018-01-01 → 2024-12-31
Base rate (y_eligible_delay): 5.3330%
Data source:   SYNTHETIC (smoke-test grade)


In [3]:
import joblib
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    average_precision_score, brier_score_loss, f1_score, roc_auc_score,
)

from src.config import ARTEFACTS_DIR
from src.eval.calibration import expected_calibration_error
from src.eval.profit_metric import ProfitConfig, total_roi
from src.models.registry import (
    make_decision_tree, make_dummy, make_logistic_regression, make_mlp,
    make_random_forest, make_xgboost,
)
from src.pipeline.build import build_pipeline
from src.pipeline.splits import temporal_split
from src.data.ec261 import KM_PER_MILE

y = df["y_eligible_delay"].to_numpy()
split = temporal_split(df)
X_tr = df.iloc[split.train_idx].reset_index(drop=True)
X_va = df.iloc[split.val_idx].reset_index(drop=True)
X_te = df.iloc[split.test_idx].reset_index(drop=True)
y_tr = y[split.train_idx]
y_va = y[split.val_idx]
y_te = y[split.test_idx]
print(f"train={len(X_tr):,}  val={len(X_va):,}  test={len(X_te):,}")
print(f"base rates: train={y_tr.mean():.4%}  val={y_va.mean():.4%}  test={y_te.mean():.4%}")


train=10,484  val=2,066  test=2,151
base rates: train=5.1316%  val=6.1955%  test=5.4858%


In [4]:
def evaluate(name, pipe, X, y_true, T, d_km, calibrated=True):
    proba = pipe.predict_proba(X)[:, 1]
    auc = roc_auc_score(y_true, proba) if y_true.sum() > 0 else float("nan")
    ap = average_precision_score(y_true, proba)
    brier = brier_score_loss(y_true, proba)
    ece = expected_calibration_error(y_true, proba)
    pred = (proba >= 0.5).astype(int)
    f1 = f1_score(y_true, pred, zero_division=0)
    roi = total_roi(y_true, proba, T, d_km, ProfitConfig())
    return {
        "model": name,
        "calibrated": calibrated,
        "ROC_AUC": auc, "PR_AUC": ap, "F1@0.5": f1,
        "Brier": brier, "ECE": ece,
        "ROI_perflight": roi["roi"],
        "n_buys": roi["n_buys"],
        "profit_eur": roi["profit_total_eur"],
    }

T_te = X_te["T_eur"].to_numpy()
d_te = X_te["DISTANCE"].to_numpy() * KM_PER_MILE
results = []
fitted_pipelines = {}


## Hyperparameter tuning — Grid / Randomized / Bayesian

Every search is configured with `ExpandingTimeSeriesSplit` (defined in `src/pipeline/splits.py`) which respects calendar order at every fold boundary — the *only* honest CV for a temporally-structured prediction task. The scoring function is `profit_scorer()` so we tune directly for per-flight ROI under τ\*(T, d).

| Model | Search class | Default budget | Justification |
|---|---|---|---|
| Logistic Regression | `GridSearchCV` | 6 fits (3 × 2) | Tiny grid; saga vs lbfgs occasionally matters with class-weighted loss |
| Random Forest | `RandomizedSearchCV` | 30 fits | 4-dim mixed space; random search dominates grid (Bergstra & Bengio 2012) |
| XGBoost | `BayesSearchCV` (scikit-optimize) | 50 fits | 7-dim continuous space; Bayes typically matches a 200-iter random search at ¼ cost |

Total: ≈ 86 search fits × 4 CV folds = ~344 fits. Budgets are configurable via env vars `RF_RANDOM_N_ITER` and `XGB_BAYES_N_ITER` for CI / quick experiments. The selected hyperparameters land in `artefacts/best_hyperparams.json`.

In [5]:
import json
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from src.eval.profit_metric import profit_scorer
from src.models.tuning import (
    LOGREG_GRID, RF_RANDOM_DIST, XGB_BAYES_SPACE, n_iter_rf, n_iter_xgb,
)
from src.pipeline.splits import ExpandingTimeSeriesSplit

cv = ExpandingTimeSeriesSplit(n_splits=4)
scoring = profit_scorer()
best_params_log = {}

def _tuned_fit(name, factory, search_space, search_cls, **search_kwargs):
    pipe = build_pipeline(factory())
    if search_cls is GridSearchCV:
        search = GridSearchCV(pipe, search_space, cv=cv, scoring=scoring,
                              n_jobs=1, refit=True, verbose=1)
    else:
        search = search_cls(pipe, search_space, cv=cv, scoring=scoring,
                            n_jobs=1, refit=True, verbose=1, **search_kwargs)
    search.fit(X_tr, y_tr)
    print(f"  {name}: best CV ROI = {search.best_score_:.4f}")
    print(f"  {name}: best params = {dict(search.best_params_)}")
    cal = CalibratedClassifierCV(estimator=search.best_estimator_,
                                 method="isotonic", cv="prefit")
    cal.fit(X_va, y_va)
    return search.best_estimator_, cal, dict(search.best_params_)


In [6]:
print("Trivial baseline: Dummy")
base_dummy = build_pipeline(make_dummy()).fit(X_tr, y_tr)
cal_dummy = CalibratedClassifierCV(estimator=base_dummy, method="isotonic",
                                    cv="prefit").fit(X_va, y_va)
results.append(evaluate("Dummy",            base_dummy, X_te, y_te, T_te, d_te, calibrated=False))
results.append(evaluate("Dummy+isotonic",   cal_dummy,  X_te, y_te, T_te, d_te, calibrated=True))
fitted_pipelines["Dummy+isotonic"] = cal_dummy

print("\nClassical baseline: Decision Tree (untuned, kept for diversity)")
base_dt = build_pipeline(make_decision_tree()).fit(X_tr, y_tr)
cal_dt  = CalibratedClassifierCV(estimator=base_dt, method="isotonic",
                                 cv="prefit").fit(X_va, y_va)
results.append(evaluate("DecisionTree",            base_dt, X_te, y_te, T_te, d_te, calibrated=False))
results.append(evaluate("DecisionTree+isotonic",   cal_dt,  X_te, y_te, T_te, d_te, calibrated=True))
fitted_pipelines["DecisionTree+isotonic"] = cal_dt


Trivial baseline: Dummy



Classical baseline: Decision Tree (untuned, kept for diversity)


In [7]:
print("Tuning Logistic Regression with GridSearchCV ...")
base, cal, params = _tuned_fit("LogReg", make_logistic_regression,
                                LOGREG_GRID, GridSearchCV)
best_params_log["LogReg"] = params
results.append(evaluate("LogReg",          base, X_te, y_te, T_te, d_te, calibrated=False))
results.append(evaluate("LogReg+isotonic", cal,  X_te, y_te, T_te, d_te, calibrated=True))
fitted_pipelines["LogReg+isotonic"] = cal


Tuning Logistic Regression with GridSearchCV ...
Fitting 4 folds for each of 6 candidates, totalling 24 fits


  LogReg: best CV ROI = -1.8056
  LogReg: best params = {'clf__C': 10.0, 'clf__penalty': 'l2', 'clf__solver': 'saga'}


In [8]:
print("Tuning Random Forest with RandomizedSearchCV ...")
base, cal, params = _tuned_fit("RandomForest", make_random_forest,
                                RF_RANDOM_DIST, RandomizedSearchCV,
                                n_iter=n_iter_rf(), random_state=0)
best_params_log["RandomForest"] = params
results.append(evaluate("RandomForest",          base, X_te, y_te, T_te, d_te, calibrated=False))
results.append(evaluate("RandomForest+isotonic", cal,  X_te, y_te, T_te, d_te, calibrated=True))
fitted_pipelines["RandomForest+isotonic"] = cal


Tuning Random Forest with RandomizedSearchCV ...
Fitting 4 folds for each of 3 candidates, totalling 12 fits


  RandomForest: best CV ROI = -2.8876
  RandomForest: best params = {'clf__n_estimators': 300, 'clf__min_samples_leaf': 200, 'clf__max_features': 'log2', 'clf__max_depth': None}


In [9]:
spw = float((y_tr == 0).sum() / max(1, (y_tr == 1).sum()))
try:
    from skopt import BayesSearchCV
    print("Tuning XGBoost with BayesSearchCV ...")
    base, cal, params = _tuned_fit(
        "XGBoost", lambda: make_xgboost(scale_pos_weight=spw),
        XGB_BAYES_SPACE, BayesSearchCV,
        n_iter=n_iter_xgb(), random_state=0,
    )
    best_params_log["XGBoost"] = params
    results.append(evaluate("XGBoost",          base, X_te, y_te, T_te, d_te, calibrated=False))
    results.append(evaluate("XGBoost+isotonic", cal,  X_te, y_te, T_te, d_te, calibrated=True))
    fitted_pipelines["XGBoost+isotonic"] = cal
except ImportError:
    print("scikit-optimize unavailable; falling back to RandomizedSearchCV for XGBoost.")
    fallback_dist = {
        "clf__n_estimators":     [200, 400, 600, 800],
        "clf__max_depth":        [4, 6, 8, 10],
        "clf__learning_rate":    [0.01, 0.03, 0.05, 0.1, 0.2],
        "clf__subsample":        [0.6, 0.8, 1.0],
        "clf__colsample_bytree": [0.6, 0.8, 1.0],
        "clf__reg_lambda":       [0.1, 1.0, 10.0],
        "clf__min_child_weight": [1, 5, 25],
    }
    base, cal, params = _tuned_fit(
        "XGBoost", lambda: make_xgboost(scale_pos_weight=spw),
        fallback_dist, RandomizedSearchCV,
        n_iter=n_iter_xgb(), random_state=0,
    )
    best_params_log["XGBoost"] = params
    results.append(evaluate("XGBoost",          base, X_te, y_te, T_te, d_te, calibrated=False))
    results.append(evaluate("XGBoost+isotonic", cal,  X_te, y_te, T_te, d_te, calibrated=True))
    fitted_pipelines["XGBoost+isotonic"] = cal
except Exception as e:
    print(f"XGBoost unavailable, skipping: {type(e).__name__}: {e}")
    print("On macOS, install libomp: `brew install libomp`")


Tuning XGBoost with BayesSearchCV ...


Fitting 4 folds for each of 1 candidates, totalling 4 fits


Fitting 4 folds for each of 1 candidates, totalling 4 fits


Fitting 4 folds for each of 1 candidates, totalling 4 fits


Fitting 4 folds for each of 1 candidates, totalling 4 fits


  XGBoost: best CV ROI = -0.7694
  XGBoost: best params = {'clf__colsample_bytree': 0.7654820824760737, 'clf__learning_rate': 0.047285819632944495, 'clf__max_depth': 7, 'clf__min_child_weight': 33, 'clf__n_estimators': 873, 'clf__reg_lambda': 8.36036138601526, 'clf__subsample': 0.5640343281570988}


In [10]:
try:
    print("Fitting MLP (untuned, for ladder completeness)")
    mlp_pipe = build_pipeline(make_mlp())
    mlp_pipe.fit(X_tr, y_tr)
    cal = CalibratedClassifierCV(estimator=mlp_pipe, method="isotonic", cv="prefit")
    cal.fit(X_va, y_va)
    results.append(evaluate("MLP", mlp_pipe, X_te, y_te, T_te, d_te, calibrated=False))
    results.append(evaluate("MLP+isotonic", cal, X_te, y_te, T_te, d_te, calibrated=True))
    fitted_pipelines["MLP+isotonic"] = cal
except Exception as e:
    print(f"MLP failed: {e}")


Fitting MLP (untuned, for ladder completeness)


In [11]:
(ARTEFACTS_DIR / "best_hyperparams.json").write_text(
    json.dumps(best_params_log, indent=2, default=str)
)
print(f"Wrote artefacts/best_hyperparams.json with {len(best_params_log)} entries.")
print(json.dumps(best_params_log, indent=2, default=str))


Wrote artefacts/best_hyperparams.json with 3 entries.
{
  "LogReg": {
    "clf__C": 10.0,
    "clf__penalty": "l2",
    "clf__solver": "saga"
  },
  "RandomForest": {
    "clf__n_estimators": 300,
    "clf__min_samples_leaf": 200,
    "clf__max_features": "log2",
    "clf__max_depth": null
  },
  "XGBoost": {
    "clf__colsample_bytree": 0.7654820824760737,
    "clf__learning_rate": 0.047285819632944495,
    "clf__max_depth": 7,
    "clf__min_child_weight": 33,
    "clf__n_estimators": 873,
    "clf__reg_lambda": 8.36036138601526,
    "clf__subsample": 0.5640343281570988
  }
}


In [12]:
results_df = pd.DataFrame(results).round(4)
results_df = results_df.sort_values("ROI_perflight", ascending=False).reset_index(drop=True)
print(results_df.to_string())
results_df.to_csv(ARTEFACTS_DIR / "model_comparison.csv", index=False)
print("\nWrote artefacts/model_comparison.csv")


                    model  calibrated  ROC_AUC  PR_AUC  F1@0.5   Brier     ECE  ROI_perflight  n_buys  profit_eur
0                   Dummy       False   0.5000  0.0549  0.0000  0.0549  0.0549         0.0000       0   -11259.67
1          Dummy+isotonic        True   0.5000  0.0549  0.0000  0.0519  0.0071         0.0000       0   -11259.67
2   DecisionTree+isotonic        True   0.5138  0.0573  0.0000  0.0519  0.0070         0.0000       0   -11259.67
3         LogReg+isotonic        True   0.5533  0.0623  0.0000  0.0518  0.0067         0.0000       0   -11259.67
4            MLP+isotonic        True   0.4999  0.0548  0.0000  0.0521  0.0101         0.0000       0   -11259.67
5   RandomForest+isotonic        True   0.5339  0.0681  0.0000  0.0518  0.0104         0.0000       0   -11259.67
6                     MLP       False   0.5074  0.0558  0.0000  0.0537  0.0325         0.0000       0   -11259.67
7        XGBoost+isotonic        True   0.5186  0.0593  0.0000  0.0521  0.0122         0

In [13]:
best_name = results_df.iloc[0]["model"]
print(f"Best by ROI: {best_name}")
if best_name in fitted_pipelines:
    joblib.dump(fitted_pipelines[best_name], ARTEFACTS_DIR / "best_model.joblib")
    print("Saved to artefacts/best_model.joblib")
else:
    cal_keys = [k for k in fitted_pipelines if k.endswith("+isotonic")]
    if cal_keys:
        joblib.dump(fitted_pipelines[cal_keys[-1]], ARTEFACTS_DIR / "best_model.joblib")
        print(f"Best-by-ROI is uncalibrated; saved last calibrated model {cal_keys[-1]} as the operational artefact.")


Best by ROI: Dummy


Best-by-ROI is uncalibrated; saved last calibrated model MLP+isotonic as the operational artefact.


## Headline numbers — drop-in for the final report

The block below renders every fill-in placeholder needed for §2 (data summary) and §4 (results table) of `reports/final_report.md`. After running this cell, copy each line to its `*fill in*` slot in the report.

In [14]:
print("=" * 72)
print(" HEADLINE NUMBERS FOR reports/final_report.md ".center(72, "="))
print("=" * 72)
print()
print(f"  [Data source]            {'REAL BTS' if USING_REAL_DATA else 'SYNTHETIC (smoke-test grade -- DO NOT cite in submission)'}")
print(f"  [N rows]                 {len(df):,}")
print(f"  [N train / val / test]   {len(X_tr):,} / {len(X_va):,} / {len(X_te):,}")
print(f"  [Date range]             {df['FL_DATE'].min().date()} -> {df['FL_DATE'].max().date()}")
print(f"  [EC261-eligible base rate (overall)]  {y.mean():.3%}")
print(f"  [EC261-eligible base rate (test)]     {y_te.mean():.3%}")
print()
if 'ARR_DELAY' in df.columns:
    raw_3h_rate = (df['ARR_DELAY'] >= 180).mean()
    print(f"  [Raw 3h+ delay rate]     {raw_3h_rate:.3%}  (compare vs eligible -- drop is the EC261 exemption)")
print()
print("  [Missingness audit (% NaN per column with any NaN)]:")
miss = df.isna().mean().sort_values(ascending=False)
for col, frac in miss[miss > 0].head(10).items():
    print(f"    {col:30s} {frac:.2%}")
print()
print("-" * 72)
print(" Model leaderboard (sorted by ROI on test set) ".center(72, "-"))
print("-" * 72)
print(results_df.to_string(index=False))
print()
if best_name in fitted_pipelines:
    best_row = results_df.iloc[0]
    print(f"  [Best model]             {best_row['model']}")
    print(f"  [Best ROC-AUC]           {best_row['ROC_AUC']:.4f}")
    print(f"  [Best PR-AUC]            {best_row['PR_AUC']:.4f}")
    print(f"  [Best Brier]             {best_row['Brier']:.4f}")
    print(f"  [Best ECE]               {best_row['ECE']:.4f}")
    print(f"  [Best ROI per flight]    {best_row['ROI_perflight']:.4f}")
    print(f"  [Best n_buys]            {best_row['n_buys']:,}")
    print(f"  [Best total profit (eur)] {best_row['profit_eur']:,.2f}")
print()
uncal_xgb = results_df[results_df['model'] == 'XGBoost']
cal_xgb   = results_df[results_df['model'] == 'XGBoost+isotonic']
if len(uncal_xgb) and len(cal_xgb):
    delta_ece  = uncal_xgb.iloc[0]['ECE']  - cal_xgb.iloc[0]['ECE']
    delta_roi  = (cal_xgb.iloc[0]['ROI_perflight'] - uncal_xgb.iloc[0]['ROI_perflight']) * 10_000
    print("  [Calibration impact on XGBoost]")
    print(f"    ECE drop:   {delta_ece:+.4f} ({delta_ece*100:+.2f} pp)")
    print(f"    ROI delta:  {delta_roi:+.1f} bps")
print()
print("=" * 72)
if not USING_REAL_DATA:
    print(" WARNING: SYNTHETIC DATA RUN -- numbers are illustrative, not citable.")
    print("   For the submission, run `python scripts/download_bts.py --years 2018 2019 2020 2021 2022 2023 2024`")
    print("   then re-execute this notebook.")
    print("=" * 72)



============= HEADLINE NUMBERS FOR reports/final_report.md =============

  [Data source]            SYNTHETIC (smoke-test grade -- DO NOT cite in submission)
  [N rows]                 14,701
  [N train / val / test]   10,484 / 2,066 / 2,151
  [Date range]             2018-01-01 -> 2024-12-31
  [EC261-eligible base rate (overall)]  5.333%
  [EC261-eligible base rate (test)]     5.486%

  [Raw 3h+ delay rate]     8.911%  (compare vs eligible -- drop is the EC261 exemption)

  [Missingness audit (% NaN per column with any NaN)]:

------------------------------------------------------------------------
------------ Model leaderboard (sorted by ROI on test set) -------------
------------------------------------------------------------------------
                model  calibrated  ROC_AUC  PR_AUC  F1@0.5  Brier    ECE  ROI_perflight  n_buys  profit_eur
                Dummy       False   0.5000  0.0549  0.0000 0.0549 0.0549         0.0000       0   -11259.67
       Dummy+isotonic        

## Calibration matters

Compare the `Brier` and `ECE` columns for each model with and without isotonic calibration. Tree-based models in particular benefit massively — XGBoost's raw scores are typically poorly calibrated, which would corrupt the EV math downstream.

The headline metric for model selection is **ROI under the per-flight threshold τ\*(T, d)** — not F1, not ROC-AUC. The model that wins on ROI is the one we promote to threshold optimisation in notebook 04.